### Data Preparation

This notebook is to prepare the data as 3 files due to the constraints of the project, this is directly tied to the original research paper that collected this dataset. One of the hotels is a resort hotel and the other is a city hotel so they were divided based on that. 
A country lookup table was created as the third file, this was done for the bias and fairness analysis.

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [2]:
df_raw = pd.read_csv('./data/hotel_bookings.csv')

print(f'dataset shape: {df_raw.shape}')
print(f'columns ({len(df_raw.columns)}): {list(df_raw.columns)}')

dataset shape: (119390, 32)
columns (32): ['hotel', 'is_canceled', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'company', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'reservation_status', 'reservation_status_date']


In [ ]:
# split intom city and resort hotel files, like in the paper
df_city = df_raw[df_raw['hotel'] == 'City Hotel'].copy().reset_index(drop=True)
df_resort = df_raw[df_raw['hotel'] == 'Resort Hotel'].copy().reset_index(drop=True)

print(f'City Hotel rows : {len(df_city):,}')
print(f'Resort Hotel rows : {len(df_resort):,}')
print(f'Total : {len(df_city) + len(df_resort):,}')

City Hotel rows : 79,330
Resort Hotel rows : 40,060
Total : 119,390


In [4]:
# create a country lookup table for geographic context for bias analysis
country_region_map = {
    'PRT': ('Southern Europe', 'Europe'),
    'GBR': ('Northern Europe', 'Europe'),
    'FRA': ('Western Europe', 'Europe'),
    'ESP': ('Southern Europe', 'Europe'),
    'DEU': ('Western Europe', 'Europe'),
    'ITA': ('Southern Europe', 'Europe'),
    'IRL': ('Northern Europe', 'Europe'),
    'BEL': ('Western Europe', 'Europe'),
    'BRA': ('South America', 'Americas'),
    'NLD': ('Western Europe', 'Europe'),
    'USA': ('North America', 'Americas'),
    'CHN': ('Eastern Asia', 'Asia'),
    'AUT': ('Western Europe', 'Europe'),
    'POL': ('Eastern Europe', 'Europe'),
    'CHE': ('Western Europe', 'Europe'),
    'RUS': ('Eastern Europe', 'Europe'),
    'NOR': ('Northern Europe', 'Europe'),
    'SWE': ('Northern Europe', 'Europe'),
    'DNK': ('Northern Europe', 'Europe'),
    'MOZ': ('Sub-Saharan Africa', 'Africa'),
    'AGO': ('Sub-Saharan Africa', 'Africa'),
    'CN':  ('Eastern Asia', 'Asia'),
    'NULL': ('Unknown', 'Unknown'),
}

# get all countries in the dataset and build lookup
all_countries = df_raw['country'].dropna().unique()
lookup_rows = []
for c in all_countries:
    region, continent = country_region_map.get(c, ('Other', 'Other'))
    lookup_rows.append({'country_code': c, 'region': region, 'continent': continent})

df_country_lookup = pd.DataFrame(lookup_rows).sort_values('country_code').reset_index(drop=True)

print(f'Country lookup table: {df_country_lookup.shape}')
df_country_lookup.head(10)

Country lookup table: (177, 3)


,country_code,region,continent
0,ABW,Other,Other
1,AGO,Sub-Saharan Africa,Africa
2,AIA,Other,Other
3,ALB,Other,Other
4,AND,Other,Other
5,ARE,Other,Other
6,ARG,Other,Other
7,ARM,Other,Other
8,ASM,Other,Other
9,ATA,Other,Other


In [5]:
# save the files 
df_city.to_csv('./data/city_hotel.csv', index=False)
df_resort.to_csv('./data/resort_hotel.csv', index=False)
df_country_lookup.to_csv('./data/country_lookup.csv', index=False)